In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import pandas as pd
import csv
import time
import re

# ---------------- CONFIG ----------------
INPUT_FILE = "facebook_followers.csv"   # Input CSV with column 'Profile Link'
OUTPUT_FILE = "facebook_about_data.csv" # Output file

chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--disable-notifications")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("detach", True)

driver = webdriver.Chrome(options=chrome_options)

# ---------------- LOGIN ----------------
driver.get("https://www.facebook.com/")
print("🔐 Please log in to Facebook...")
input("👉 Press Enter after you have logged in: ")

# ---------------- READ INPUT ----------------
df = pd.read_csv(INPUT_FILE)
if "Profile Link" not in df.columns:
    raise Exception("⚠ CSV must contain a column named 'Profile Link'.")
profile_links = df["Profile Link"].dropna().unique().tolist()
print(f"✅ Loaded {len(profile_links)} profile URLs.\n")

# ---------------- PREPARE OUTPUT ----------------
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "Profile URL",
        "Name",
        "Gender",
        "Birth Date",
        "Birth Year",
        "Account Creation Date",
        "Work",
        "Current City",
        "Hometown",
        "Relationship"
    ])

# ---------------- SCRAPER FUNCTION ----------------
def scrape_profile(url):
    data = {
        "Profile URL": url,
        "Name": "",
        "Gender": "",
        "Birth Date": "",
        "Birth Year": "",
        "Account Creation Date": "",
        "Work": "",
        "Current City": "",
        "Hometown": "",
        "Relationship": ""
    }

    try:
        # --- Step 1: About Overview ---
        about_url = url + ("&sk=about" if "?id=" in url else "?sk=about")
        driver.get(about_url)
        time.sleep(4)

        # --- Name ---
        try:
            name_elem = driver.find_element(By.XPATH, "//h1 | //h2//span[contains(@dir,'auto')]")
            data["Name"] = name_elem.text.strip()
        except:
            pass

        # --- Overview section ---
        elements = driver.find_elements(By.XPATH, "//div[@role='main']//span")
        texts = [el.text.strip() for el in elements if el.text.strip()]

        for text in texts:
            lower = text.lower()

            # Work
            if any(k in lower for k in [
                "works at", "worked at", "self-employed", "freelancer", "owner at", "manages"
            ]):
                data["Work"] += text + " | "

            # Current City
            elif "lives in" in lower:
                data["Current City"] = text.replace("Lives in", "").replace("lives in", "").strip()

            # Hometown
            elif lower.startswith("from "):
                data["Hometown"] = text.replace("From", "").replace("from", "").strip()

            # Relationship
            elif any(k in lower for k in [
                "single", "married", "relationship", "engaged", "divorced", "widowed"
            ]):
                data["Relationship"] = text.strip()

        # --- Step 2: Contact & Basic Info ---
        try:
            contact_url = url + ("&sk=about_contact_and_basic_info" if "?id=" in url else "?sk=about_contact_and_basic_info")
            driver.get(contact_url)
            time.sleep(4)

            spans = driver.find_elements(By.XPATH, "//span[@dir='auto']")
            for span in spans:
                txt = span.text.strip()
                lower = txt.lower()

                # Gender
                if lower in ["male", "female", "custom"]:
                    data["Gender"] = txt

                # Birth Date
                elif any(month in lower for month in [
                    "january", "february", "march", "april", "may", "june",
                    "july", "august", "september", "october", "november", "december"
                ]):
                    data["Birth Date"] = txt
                    year_match = re.search(r'\b(19|20)\d{2}\b', txt)
                    if year_match:
                        data["Birth Year"] = year_match.group()

                # Birth Year only
                elif txt.isdigit() and len(txt) == 4 and txt.startswith(('19', '20')):
                    if not data["Birth Year"]:
                        data["Birth Year"] = txt
        except Exception as e:
            print("⚠ Could not open Contact & Basic Info section:", e)

        # --- Step 3: Profile Transparency (Account Creation Date) ---
        try:
            trans_url = url + ("&sk=about_profile_transparency" if "?id=" in url else "?sk=about_profile_transparency")
            driver.get(trans_url)
            time.sleep(5)

            date_text = ""
            elements = driver.find_elements(By.XPATH, "//div[@role='main']//span[@dir='auto']")

            for idx, el in enumerate(elements):
                text = el.text.strip()
                if text.lower() == "creation date":
                    # previous element usually has the actual date (e.g. 8 August 2016)
                    if idx > 0:
                        possible_date = elements[idx - 1].text.strip()
                        if re.search(r'\d{1,2}\s+\w+\s+\d{4}', possible_date):
                            date_text = possible_date
                            break

            # fallback: any standalone date text
            if not date_text:
                for el in elements:
                    txt = el.text.strip()
                    if re.search(r'\d{1,2}\s+\w+\s+\d{4}', txt):
                        date_text = txt
                        break

            data["Account Creation Date"] = date_text

        except Exception as e:
            print("⚠ Could not extract transparency info:", e)

        print(f"  ✅ Extracted: Gender={data['Gender']} | Birth={data['Birth Date']} | Year={data['Birth Year']} | Joined={data['Account Creation Date']}")
        print(f"     🏙️ {data['Current City']} | 🏡 {data['Hometown']}")
        return data

    except Exception as e:
        print(f"⚠ Error scraping {url}: {e}")
        return data


# ---------------- MAIN LOOP ----------------
for i, url in enumerate(profile_links, start=1):
    print(f"\n📄 [{i}/{len(profile_links)}] Scraping: {url}")
    info = scrape_profile(url)

    with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            info["Profile URL"],
            info["Name"],
            info["Gender"],
            info["Birth Date"],
            info["Birth Year"],
            info["Account Creation Date"],
            info["Work"].strip(" | "),
            info["Current City"],
            info["Hometown"],
            info["Relationship"]
        ])

    time.sleep(3)

print("\n✅ DONE — Data saved to:", OUTPUT_FILE)
driver.quit()

🔐 Please log in to Facebook...


👉 Press Enter after you have logged in:  


✅ Loaded 2099 profile URLs.


📄 [1/2099] Scraping: https://www.facebook.com/profile.php?id=100087433700818
  ✅ Extracted: Gender= | Birth= | Year= | Joined=
     🏙️ Lillian, Texas | 🏡 Arlington, Texas

📄 [2/2099] Scraping: https://www.facebook.com/profile.php?id=61581384073915
  ✅ Extracted: Gender= | Birth= | Year= | Joined=
     🏙️ Houston, Texas | 🏡 Cambridge, Massachusetts

📄 [3/2099] Scraping: https://www.facebook.com/profile.php?id=100093213115387
  ✅ Extracted: Gender= | Birth=4 August | Year=1974 | Joined=10 June 2023
     🏙️  | 🏡 

📄 [4/2099] Scraping: https://www.facebook.com/profile.php?id=100064229881516
  ✅ Extracted: Gender= | Birth= | Year= | Joined=17 September 2020
     🏙️  | 🏡 

📄 [5/2099] Scraping: https://www.facebook.com/profile.php?id=61581237952046
  ✅ Extracted: Gender= | Birth= | Year= | Joined=
     🏙️  | 🏡 

📄 [6/2099] Scraping: https://www.facebook.com/profile.php?id=100070743042928
  ✅ Extracted: Gender= | Birth= | Year= | Joined=11 February 2018
     🏙️  |